# Entscheidungsbäume und Ensemble-Methoden

In diesem Notebook behandeln wir die folgenden Themen:

1. **Entscheidungsbäume**: Aufbau anhand einer Metrik, Beschneiden anhand einer anderen, Modellauswahl
2. **Ensemble-Methoden**: Bootstrap, Bagging, Random Forests

Der Datensatz, auf dem wir uns konzentrieren, ist **UCI Heart Disease Dataset (Cleveland-Subset)**. Er umfasst 303 Patienten, 13 klinische Merkmale und eine binäre Zielvariable (Herzerkrankung vorhanden / nicht vorhanden).

## Benötigte Module

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, plot_tree

## Reproduzierbarkeit

In [ ]:
seed = 19751979

## Datensatz zu Herzerkrankungen laden

Der Datensatz zu Herzerkrankungen ([Cleveland Heart Disease](https://archive.ics.uci.edu/dataset/45/heart+disease)) aus dem UCI ML Repository enthält medizinische Messungen von 303 Patienten. Jede Zeile repräsentiert einen Patienten, der durch 13 klinische Merkmale beschrieben wird. Die Zielvariable gibt das Vorhandensein (`1`) oder Fehlen (`0`) einer Herzerkrankung an. Die folgende Tabelle beschreibt alle Spalten der Merkmalmatrix:

| Originalname | Merkmal | Beschreibung    | Datentyp |
|--------------|---------|-----------------|----------|
| `age`      | Alter     | Alter in Jahren | numerisch |
| `sex`      | Geschlecht | 0 = weilblich, 1 = männlich | binär |
| `cp`       | Brustschmerzen | Brustschmerzentyp | kategorial |
| `trestbps` | Ruheblutdruck | Ruheblutdruck (mm Hg) | numerisch |
| `chol`     | Serumcholesterin | Serumcholesterin (mg/dl) | numerisch |
| `fbs`      | Erhöhter Blutzucker | Nüchternblutzucker > 120 mg/dl | binär |
| `restecg`  | Ruhe-EKG | Ruhe-EKGs | kategorial |
| `thalach`  | Maximale Herzfrequenz | Maximale Herzfrequenz erreicht | numerisch |
| `exang`    | Belastungsangina | Belastungsinduzierte Angina | binär |
| `oldpeak`  | ST-Depression | Belastungsinduzierte ST-Senkung | numerisch |
| `slope`    | ST-Segment | Steigung des ST-Segments bei Belastung | kategorial |
| `ca`       | Gefärbte Hauptgefäße | Große Gefäße, mittels Fluoroskopie eingefärbt (0–3) | numerisch |
| `thal`     | Thallium-Herzscan | Nachweis von Ischämie oder Narbengewebe | kategorial |

<span style="font-size:x-small">Weitere Informationen zum Datensatz sind in der folgenden Publikation verfügbar:<br />
R. Detrano et al. [International application of a new probability algorithm for the diagnosis of coronary artery disease](https://www.ajconline.org/article/0002-9149(89)90524-9/pdf). American Journal of Cardiology (1989)</span>

In [ ]:
def parse_dataset(lines: list[str]) -> tuple[np.ndarray, np.ndarray, dict]:
    """
    Parsiert den Datensatz zur Rotweinqualität aus Text.
    :param lines: Inhalt der Textdatei als `list` von `str`-Instanzen, eine pro Zeile.
    :return: Tupel der Form `(x, y, features)`, wobei:
             - `x` Merkmalmatrix als Array der Form `(Beobachtungen, Merkmale)` ist;
             - `y` Ausgabevektor als binärer Array der Form `(Beobachtungen)` ist;
             - `features` Merkmaldefinitionen als `dict`-Object ist, in dem jeder
                  Schlüssel ein Merkmalname ist, und sein Wert - `list` mit Kategorienamen oder
                  einelementige Liste mit der Messeinheit ist.
    """
    delimiter = ','
    columns = {
        'Alter': ['Jahre'],
        'Geschlecht': ['weiblich', 'mänlich'],
        'Brustschmerzen': ['typische Angina', 'atypische Angina', 'nicht-anginös', 'asymptomatisch'],
        'Ruheblutdruck': ['mm Hg'],
        'Serumcholesterin': ['mg/dl'],
        'Erhöhter Blutzucker': ['nein', 'ja'],
        'Ruhe-EKG': ['normal', 'ST-T', 'LVH'],
        'Maximale Herzfrequenz': ['S/min'],
        'Belastungsangina': ['nein', 'ja'],
        'ST-Depression': ['k.A.'],
        'ST-Segment': ['ansteigend', 'flach', 'abfallend'],
        'Gefärbte Hauptgefäße': ['Anzahl'],
        'Thallium-Herzscan': ['normal', 'behobener Defekt', 'reversibler Defekt']
    }
    category_counts = [len(v) for v in columns.values()]
    observation_count, feature_count = len(lines), len(columns)
    x = np.full((observation_count, feature_count), np.nan, dtype=np.float64)
    y = np.empty(observation_count, dtype=np.uint16)
    for i, line in enumerate(lines):
        values = line.strip('\r\n').split(delimiter)
        if len(values) != feature_count + 1:
            raise ValueError(f'Unerwartete Anzahl an Spalten in Zeile {i + 1}')
        for j, value in enumerate(values):
            if j < feature_count:
                if value != '?':
                    if category_counts[j] == 1:
                        x[i, j] = float(value)
                    else:
                        value = int(float(value))
                        if j == 2 or j == 10:
                            value -= 1
                        elif j == 12:
                            value = [3, 6, 7].index(value)
                        if not (0 <= value < category_counts[j]):
                            txt = f'{list(columns.keys())[j]} in Zeile {i + 1}'
                            raise ValueError(f'Unerwartete Kategorieindex für {txt}')
                        x[i, j] = value
            else:
                y[i] = int(value)

    # Zielvariable binarisieren: 0 = keine Erkrankung, 1 = Erkrankung
    y[y != 0] = 1
    return x, y, columns


def extract_lines_from_url(dataset_path: str) -> list[str]:
    """
    Lädt einen Datensatz vom UCI-Repository herunter.
    :param dataset_path: URL zum Datensatz; relativ zum Datenbanken-Ordner.
    :return: Inhalt des Datensatzes als `list` von `str`-Instanzen, eine pro Zeile.
    """
    import urllib
    uci_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/'
    response = urllib.request.urlopen(uci_url + dataset_path)
    return response.read().decode('utf-8').splitlines()


def extract_lines_from_file(file_path: str) -> list[str]:
    """
    Lädt den Datensatz zur Rotweinqualität von lokaler Datei (.csv.gz).
    :return: Inhalt des Datensatzes als `list` von `str`-Instanzen, eine pro Zeile.
    """
    import gzip
    function_open = gzip.open if file_path.endswith('.gz') else open
    with function_open(file_path, 'rt', encoding='utf-8') as file_handle:
        result = file_handle.readlines()
    return result


def print_feature_info(x: np.ndarray, features: dict[str, list]):
    """
    Gibt die Namen und Datentypen aller Features aus.

    :param x: Merkmalmatrix als Array der Form `(Beobachtungen, Merkmale)`.
    :param features: Merkmaldefinitionen als `dict`-Object.
    """
    print("\nMerkmaltypen\n------------")
    for i, (feature_name, feature_info) in enumerate(features.items()):
        category_count = len(feature_info)
        if category_count == 1:
            txt = f'numerisch, [{x[:, i].min()}, {x[:, i].max()}]'
        else:  # type(feature_info) is str:
            txt = f'{category_count} Kategorien: ' + ', '.join(feature_info)
        print(f'{feature_name:>22s} | {txt}')


def print_missing_values(x: np.ndarray, features: dict[str, list]):
    """
    Gibt eine Zusammenfassung der fehlenden Werte pro Merkmal aus.
    
    :param x: Merkmalmatrix als Array der Form `(Beobachtungen, Merkmale)`.
    :param features: Merkmaldefinitionen als `dict`-Object.
    """
    print('\nFehlende Werte\n--------------')
    for i, feature_name in enumerate(features.keys()):
        missing_count = np.isnan(x[:, i]).sum().item()
        print(f'{feature_name:>22s}: {missing_count}')


def print_patient(x: np.ndarray, features: dict[str, list]):
    """
    Gibt alle Merkmalswerte für einen Patienten aus.
    
    :param x: Merkmalvektor für den Patienten / die Patientin.
    :param features: Merkmaldefinitionen als `dict`-Object.
    """
    for (feature_name, categories), value in zip(features.items(), x):
        if len(categories) == 1:
            value = f'{value} {categories[0]}'
        else:  # len(categories) > 1
            value = categories[int(value)]
        print(f'{feature_name:>22s}: {value}')


dataset_url = 'heart-disease/processed.cleveland.data'
x, y, features = parse_dataset(extract_lines_from_url(dataset_url))

print(f'Dimensionen der Markmalsmatrix: {x.shape}')
print(f'Dimensionen des Ausgabevektors: {y.shape}')
print(f'                  Merkmalnamen: {','.join(features.keys())}')
print(f'            Erster Ausgabewert: {y[0]}')
print(f'                   Erste Zeile: {x[0, :]}')

In [ ]:
print(features)  # {Merkmalname: 1. [Messeinheit] oder 2. [Kategoriennamen]}

# Datenvorverarbeitung

## Kodierung kategorialer Merkmale

Mehrere Spalten sind **kategorial**, liegen jedoch als Zahlen vor: `Brustschmerzen`, `Ruhe-EKG`,  `ST-Segment` und `Thallium-Herzscan`. Die Werte 0, 1, 2 und 3 in der Spalte `Brustschmerzen` weisen keine sinnvolle Reihenfolge auf – sie dienen lediglich als Bezeichnungen für verschiedene Arten von Brustschmerzen.

Das **One-Hot-Kodierung** erzeugt für jeden Kategoriewert eine eigene binäre Spalte und vermeidet so die fälschliche Implikation, dass 3 > 2 > 1 > 0 gilt.

Zudem werden wir die Spalten `Geschlecht`, `Erhöhter Blutzucker` und `Belastungsangina` explizit als binär behandeln (sie liegen bereits im Format 0/1 vor); für diese ist daher keine Kodierung erforderlich.

### Übung: One-Hot-Kodierung kategorialer Spalten

Wenden Sie ausgehend von `x` das **One-Hot-Kodierung** auf die Spalten `['Brustschmerzen', 'Ruhe-EKG', 'ST-Segment', 'Thallium-Herzscan']` an. Entfernen Sie die ursprünglichen Spalten.

Speichern Sie das Ergebnis in `x_encoded`.

In [ ]:
x_encoded = []
feature_names_encoded = []

for feature_index, (feature_name, info) in enumerate(features.items()):
    category_count = len(info)
    if category_count <= 2:
        # Reeles oder binäres Merkmal
        x_current = x[:, [feature_index]]
        feature_names_encoded.append(feature_name)
    else:
        # Kategoriales Merkmal mit mehr als 2 Kategorien
        x_current = np.full((x.shape[0], category_count), 0, dtype=x.dtype)
        for category_index, category_name in enumerate(info):
            x_current[x[:, feature_index] == category_index, category_index] = 1
            feature_names_encoded.append(feature_name + ' ' + category_name)
        x_current[np.isnan(x[:, feature_index]), :] = np.nan
    x_encoded.append(x_current)

x_encoded = np.hstack(tuple(x_encoded))
print(f' Form vor der Kodierung: {x.shape}')
print(f'Form nach der Kodierung: {x_encoded.shape}')
print(f'Merkmalnamen nach der Kodierung:')
print(', '.join(feature_names_encoded))
del feature_index, feature_name, info, category_count, x_current, category_index, category_name

## Vorbereitung der Daten für die Modellierung

Wir haben eine Feature-Matrix $X_{encoded}$ und einen Zielvektor $y$ erstellt. Jetzt können wir diese in Trainings- und Testdatensätze aufteilen.

Da wir heute ausschließlich mit baum-basierten Methoden arbeiten, verzichten wir auf die Feature-Skalierung. Entscheidungsbäume sind invariant gegenüber monotonen Transformationen der Features.

In [ ]:
# Trainings-/Test-Aufteilung (stratifiziert zur Wahrung der Klassenbalance)
x_train, x_test, y_train, y_test =\
    train_test_split(x_encoded, y, test_size=0.2, random_state=seed, stratify=y)

print(f'Trainingssatz: {x_train.shape[0]} Beobachtungen')
print(f'     Testsatz: {x_test.shape[0]} Beobachtungen')
print(f'Klassenverteilung (Training): {np.bincount(y_train) / y_train.size}')
print(f'    Klassenverteilung (Test): {np.bincount(y_test) / y_test.size}')

---
# Entscheidungsbäume – Aufbauen, Beschneiden, Auswählen

In früheren Notebooks haben wir Entscheidungsbäume trainiert, indem wir einen einzelnen Hyperparameter
(z. B. `max_depth`) variiert haben. Heute lernen wir einen methodischeren Arbeitsablauf kennen:

1. **Aufbau** eines vollständigen (unbeschnittenen) Baumes unter Verwendung eines Split-Kriteriums (Gini oder Entropie).
2. **Beschneiden** des Baumes mittels **Kosten-Komplexitäts-Beschneidung** (`ccp_alpha`), wodurch eine
ganze Familie von Bäumen erzeugt wird – vom voll ausgewachsenen Baum bis hin zum bloßen Stumpf.
3. **Auswahl** des besten Baumes mithilfe eines Validierungsdatensatzes oder durch Kreuzvalidierung.

Dies trennt das Aufbaukriterium vom Kriterium für das Beschneiden bzw. die Auswahl.

## Teil 1: Aufbau vollständiger (ungeschnittener) Entscheidungsbäume

In [ ]:
full_trees = {}
for criterion_name, criterion_code in {'Gini': 'gini', 'Entropie': 'entropy'}.items():
    # Einen vollständig expandierten Baum mit dem ausgewählten Kriterium erstellen
    tree = DecisionTreeClassifier(criterion=criterion_code, random_state=seed)
    tree.fit(x_train, y_train)
    full_trees[criterion_name] = tree
    print(f'Baum mit {criterion_name}')
    print(f'               Tiefe: {tree.get_depth()}')
    print(f'             Blätter: {tree.get_n_leaves()}')
    print(f'Training-Genauigkeit: {tree.score(x_train, y_train):.1%}')
    print(f'    Test-Genauigkeit: {tree.score(x_test, y_test):.1%}')
    print()
    
del criterion_name, criterion_code, tree

Beide Bäume weisen eine Trainingsgenauigkeit von 100 % auf – sie haben die Trainingsdaten perfekt auswendig gelernt. Die Testgenauigkeit ist jedoch deutlich geringer. Dies ist ein klassisches Beispiel für Überanpassung, und das Pruning dient dazu, dieses Problem zu beheben.

## Pfad des Cost-Complexity-Prunings

Die Kosten-Komplexitäts-Verlustfunktion für einen Entscheidungsbaum $T$ ist definiert als:

$R_\alpha(T) = R(T) + \alpha |T|$

Dabei gilt:

- $R(T)$ ist der Trainingsfehler (oder die Unreinheit) des Baumes, typischerweise berechnet als die Summe der Fehlklassifikationen oder der quadratischen Fehler über alle Blattknoten hinweg.
- $\alpha$ (Alpha) ist der Komplexitätsparameter ($\alpha \ge 0$), der die Baumgröße bestraft.
- $|T|$ ist die Anzahl der Blattknoten im Baum.

Die Methode [`cost_complexity_pruning_path`](https://scikit-learn.org/stable/auto_examples/tree/plot_cost_complexity_pruning.html) von einem Entscheidungsbaum berechnet die **effektiven $\alpha$-Werte**, bei denen Teilbäume beschnitten werden, sowie die Gesamtunreinheit der Blätter in jedem Schritt.

Mit zunehmendem `ccp_alpha` werden mehr Knoten beschnitten, also wir erstellen einfachere Bäume.

In [ ]:
def plot_impurities(prunings: dict) -> plt.Figure:
    """
    Visualisiert die Werte der Verlustfunktion eines Baums in Bezug auf die Alpha-Parameterwerte.
    :param prunings: Mapping der Form `{Kostenkriterium: Pruning-Pfad}`.
    :return: das neu erstellte Figure-Objekt.
    """
    figure, ax = plt.subplots(figsize=(6, 4), dpi=100)
    for i, (criterion_name, pruning_info) in enumerate(prunings.items()):
        alphas = pruning_info.ccp_alphas
        impurities = pruning_info.impurities
        ax.plot(alphas, impurities, '.--', label=criterion_name)
    ax.set(axisbelow=True, xlabel='$\\alpha$', ylabel='Gesamtverunreinigung')
    ax.set(ylim=[0, 1])
    ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
    ax.legend()
    return figure


prunings = {
    criterion_name: tree.cost_complexity_pruning_path(x_train, y_train)
    for criterion_name, tree in full_trees.items()}
figure = plot_impurities(prunings)
figure.tight_layout()
plt.show(figure)

### Übung: Trainings- und Testgenauigkeit entlang des Pruning-Pfades

Fokusieren wir uns auf das **Gini**-Kriterium. Im Code-Abschnitt unten:

1. Ein [`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) wird für jedes $\alpha$ in `prunings['gini']` wird trainiert.
2. **Trainings-** und **Testgenauigkeit** von jedem Baum werden erfasst.
3. Stellen Sie beide Kurven auf denselben Achsen dar ($x$ = `ccp_alpha`, $y$ = Genauigkeit).

Dies veranschaulicht die [Verzerrung-Varianz-Dilemma](https://de.wikipedia.org/wiki/Verzerrung-Varianz-Dilemma) in Abhängigkeit von der Pruning-Stärke.

In [ ]:
alphas = prunings['Gini'].ccp_alphas
accuracies = np.empty((alphas.size, 2), dtype=np.float64)  # (alphas, [train, test])
for i, alpha in enumerate(alphas):
    tree = DecisionTreeClassifier(criterion='gini', ccp_alpha=alpha, random_state=seed)
    tree.fit(x_train, y_train)
    accuracies[i, 0] = tree.score(x_train, y_train)
    accuracies[i, 1] = tree.score(x_test, y_test)

figure, ax = plt.subplots(figsize=(6, 4), dpi=100)
ax.plot(alphas, accuracies[:, 0], '.--', label='Trainig')
ax.plot(alphas, accuracies[:, 1], '.--', label='Test')
ax.set(axisbelow=True, xlabel='$\\alpha$', ylabel='Genauigkeit')
ax.set(ylim=[0, 1])
ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
ax.legend()
figure.tight_layout()
plt.show(figure)

## Modellselektion

Wir haben gesehen, wie sich die Genauigkeit beim Pruning (Beschneiden) verändert. Nun wollen wir das **optimale $\alpha$** mithilfe zweier Ansätze gezielt auswählen.

### Ansatz A — Validierungsdatensatz

Wir werden die Trainingsdaten weiter in einen Trainings- und einen Validierungsdatensatz unterteilen. So können wir den $\alpha$-Wert wählen, der die höchste **Validierungsgenauigkeit** aufweist.

In [ ]:
def plot_tree_accuracies(aplhas: np.ndarray, accuracies: np.ndarray) -> plt.Figure:
    """
    Visualisiert die Validierungsgenauigkeiten für alle Alpha-Werte.
    :param alphas: Alle Werte des Parameters Alpha als Array der Form `(k,)`.
    :param accuracies: Genauigkeiten als Array der Form:
                       - `(k,)` wenn sie mit der Hilfe eines Validierungssatzes berechnet wurden
                       - `(k, 2)` wenn sie mittels Kreuzvalidierung berechnet wurden; Spalte 0 muss
                          den Mittelwert, Spalte 1 - eine Standardabweichung beinhalten.
    :return: das neu erstellte Figure-Objekt.    
    """
    figure, ax = plt.subplots(figsize=(7, 4), dpi=100)
    if len(accuracies.shape) == 1:
        # Genauigkeiten-Vektor: Linie zeichnen
        ax.plot(alphas, accuracies, '.-', color='steelblue')
        best_alpha = alphas[np.argmax(accuracies)]
        ylabel = 'Validierungsgenauigkeit'
    else:
        # Genauigkeiten-Matrix: Linie + schattierten Bereich zeichnen
        ax.plot(alphas, accuracies[:, 0], '.-', color='steelblue')
        y_bottom = accuracies[:, 0] - accuracies[:, 1]
        y_top = accuracies[:, 0] + accuracies[:, 1]
        ax.fill_between(alphas, y_bottom, y_top, alpha=0.2, color='steelblue')
        best_alpha = alphas[np.argmax(accuracies[:, 0])]
        ylabel = 'CV Genauigkeit'
    txt = f'$\\alpha = {best_alpha:.4f}$'
    ax.axvline(best_alpha, color='red', linestyle='--', label=txt)
    ax.set(axisbelow=True, xlabel='$\\alpha$', ylabel=ylabel)
    ax.set(xlim=[alphas.min(), alphas.max()])
    ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
    ax.legend()
    figure.tight_layout()
    return figure
    

# Trainingsdaten in Trainings- und Validierungssatz aufteilen
x_train2, x_val, y_train2, y_val =\
    train_test_split(x_train, y_train, test_size=0.25, random_state=seed, stratify=y_train)

# Gini-Bäume entlang des Prunings-Pfades auf dem kleineren Trainingssatz erstellen
tree = DecisionTreeClassifier(criterion='gini', random_state=seed)
tree.fit(x_train2, y_train2)
alphas = tree.cost_complexity_pruning_path(x_train2, y_train2).ccp_alphas
accuracies = np.empty_like(alphas, dtype=np.float64)
for i, alpha in enumerate(alphas):
    t = DecisionTreeClassifier(criterion='gini', ccp_alpha=alpha, random_state=seed)
    t.fit(x_train2, y_train2)
    accuracies[i] = t.score(x_val, y_val)

# Genauigkeiten beim Validierungssatz visualisieren
figure = plot_tree_accuracies(alphas, accuracies)
plt.show(figure)

# Den besten Baum bestimmen
index_best_alpha = np.argmax(accuracies)
best_alpha = alphas[index_best_alpha]
print(f'      Bester Alpha-Wert: {best_alpha:.4f}')
print(f'Validierungsgenauigkeit: {accuracies[index_best_alpha]:.1%}')

### Ansatz B: 5-fache Kreuzvalidierung

Wenden wir nun eine **5-fache stratifizierte Kreuzvalidierung** auf den vollständigen Trainingsdatensatz (`x_train`, `y_train`) an, um das beste $\alpha$ auszuwählen.

Für jedes Alpha wird der **Mittelwert** und die **Standardabweichung** der CV-Scores erfasst. Dafür wird die Funktion
[`cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) mit einem vordefinierten [`StratifiedKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)-Objekt verwendet.

Danach wird das Alpha mit dem höchsten mittleren CV-Score ausgewählt.

Schließlich wird der Mittelwert ± Standardabweichung in Abhängigkeit von $\alpha$ grafisch dargestellt.

In [ ]:
# Kreuzvalidierungsobjekt definieren
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

# Gini-Bäume entlang des Prunings-Pfades auf dem Trainingssatz erstellen
tree = DecisionTreeClassifier(criterion='gini', random_state=seed)
tree.fit(x_train, y_train)
alphas = tree.cost_complexity_pruning_path(x_train, y_train).ccp_alphas
accuracies = np.empty((alphas.size, 2), dtype=np.float64)  # (alphas, [mean, sd])
for i, alpha in enumerate(alphas):
    t = DecisionTreeClassifier(criterion='gini', ccp_alpha=alpha, random_state=seed)
    scores = cross_val_score(t, x_train, y_train, cv=cv, scoring='accuracy')
    accuracies[i, 0] = scores.mean()
    accuracies[i, 1] = scores.std()

# Kreuzvalidierungseinschätzungen visualisieren
figure = plot_tree_accuracies(alphas, accuracies)
plt.show(figure)

# Den besten Baum bestimmen
index_best_alpha = np.argmax(accuracies[:, 0])
best_alpha = alphas[index_best_alpha]
print(f'Bester Alpha-Wert: {best_alpha:.4f}')
print(f'   CV-Genauigkeit: {accuracies[index_best_alpha, 0]:.3f} ± {accuracies[index_best_alpha, 1]:.3f}')

## Der siegreiche Entscheidungsbaum

Trainieren wir den endgültigen Baum mit dem besten Alpha aus der Kreuzvalidierung, evaluieren wir ihn auf dem **Testsatz** (den wir während der Auswahl nicht angerührt haben) und visualisieren wir ihn.

In [ ]:
# Den finalen Baum mit dem besten CV-Alpha trainieren
final_tree =\
    DecisionTreeClassifier(criterion='gini', ccp_alpha=best_alpha, random_state=seed)
final_tree.fit(x_train, y_train)

accuracy_cv_tree = final_tree.score(x_test, y_test)
accuracy_full_tree = full_trees['Gini'].score(x_test, y_test)
print(f'CV-Baum — Tiefe: {final_tree.get_depth()}, Blätter: {final_tree.get_n_leaves()}')
print(f'Testgenauigkeit: {accuracy_cv_tree:.3f}')
print(f'Testgenauigkeit des kompletten Baums: {accuracy_full_tree:.3f}')

In [ ]:
# Den beschnittenen Baum visualisieren
figure, ax = plt.subplots(figsize=(20, 10))
plot_tree(final_tree, feature_names=feature_names_encoded, class_names=['gesund', 'erkrankt'],
          filled=True, rounded=True, ax=ax, fontsize=8, impurity=True)
figure.tight_layout()
plt.show(figure)

**Fragen:**
- Ist der beschnittene Baum leichter zu interpretieren als der vollständige?
- Betrachten Sie die oberen Aufteilungen – welche Merkmale sind für den Baum am aussagekräftigsten?
- Lässt sich der beschnittene Baum besser generalisieren (höhere Testgenauigkeit) als der unbeschnittene?

In [ ]:
y_pred_tree = final_tree.predict(x_test)
cm = confusion_matrix(y_test, y_pred_tree)
disp = ConfusionMatrixDisplay(cm, display_labels=['gesund', 'erkrankt'])
figure, ax = plt.subplots(figsize=(3.6, 2.6), dpi=100)
disp.plot(ax=ax, cmap='Blues')
figure.tight_layout()
plt.show(figure)

---
# Vom einzelnen Baum zum Wald

Anstatt alles auf einen einzigen Entscheidungsbaum zu setzen: Was wäre, wenn wir *viele* Bäume trainieren und sie **abstimmen** ließen? Dies ist das Kernprinzip der **Ensemble-Methoden**.

Heute nähern wir uns Schritt für Schritt den **Random Forests** an:

1. **Bootstrap** — Stichprobenziehung mit Zurücklegen
2. **Bagging** — Training mehrerer Modelle auf Bootstrap-Stichproben, anschließend Abstimmung
3. **Random Forest** — Bagging + zufällige Teilmengen von Merkmalen
4. **Out-of-Bag-Fehler** — kostenlose Validierung, kein separates Set erforderlich
5. **Merkmalsbedeutung** — auf welche Merkmale stützt sich der Wald am stärksten?

## Bootstrap-Sampling

**Bootstrap** = Ziehen von $n$ Stichproben aus einem Datensatz der Größe $n$ mit Zurücklegen.

Einige Stichproben werden mehrfach auftreten, andere gar nicht. Im Durchschnitt erscheinen etwa 63,2 % der ursprünglichen Stichproben in jeder Bootstrap-Stichprobe. Der Rest – etwa 36,8 % – ist OOB ("Out-of-Bag").

Schauen wir uns das in der Praxis an!

In [ ]:
generator = np.random.default_rng(seed)
n = x_train.shape[0]
bootstrap_indices = generator.choice(n, size=n, replace=True)
unique_in_sample = len(set(bootstrap_indices))
oob_size = n - unique_in_sample

print(f'Größe des Trainingsdatensatzes: {n}')
print(f'    Bootstrap-Stichprobengröße: {len(bootstrap_indices)}')
print(f'    Einzigartige Beobachtungen: {unique_in_sample} ({unique_in_sample/n*100:.1f}%)')
print(f'               Erwartungsgröße: {n*(1 - (1 - 1 / n)**n):.0f} (≈ 63.2%)')
print(f'             OOB Beobachtungen: {oob_size} ({(oob_size) / n * 100:.1f}%)')

### Übung: Bootstrap-Abdeckungs-Experiment

Wiederholen Sie das Bootstrap-Sampling 200 Mal. Notieren Sie für jede Wiederholung, welcher Anteil der ursprünglichen Trainingsdaten in der Bootstrap-Stichprobe enthalten ist.

Erstellen Sie anschließend ein Histogramm dieser Anteile. Liegt das Zentrum der Verteilung bei 63,2 %?

Berechnen Sie außerdem den Mittelwert der 200 Anteile.

In [ ]:
bootstrap_count = 200
coverage_fractions = np.empty(bootstrap_count, dtype=np.float64)
n = x_train.shape[0]
for i in range(bootstrap_count):
    bootstrap_indices = generator.choice(n, size=n, replace=True)
    coverage_fractions[i] = len(set(bootstrap_indices)) / n

coverage_mean = coverage_fractions.mean()
figure, ax = plt.subplots(figsize=(6, 3), dpi=100)
ax.hist(coverage_fractions, bins=25, edgecolor='white')
ax.axvline(coverage_mean, color='red', label=f'Mean = {coverage_mean:.3f}')
ax.axvline(0.632, color='orange', linestyle=':', linewidth=2, label='Erwartung')
ax.set(axisbelow=True, xlabel='Anteil eindeutiger Beobachtungen', ylabel='Anzahl')
ax.grid(axis='y', color='#A0A0A0', linestyle='--', linewidth=0.5)
ax.legend()
figure.tight_layout()
plt.show(figure)
del bootstrap_count, coverage_fractions, n, i, bootstrap_indices
del coverage_mean, figure, ax

## Bagging — Bootstrap Aggregating

**Bagging** kombiniert das Bootstrap-Sampling mit dem Modelltraining:

1. Eine Bootstrap-Stichprobe aus den Trainingsdaten ziehen.
2. Einen Entscheidungsbaum auf dieser Stichprobe trainieren.
3. Diesen Vorgang viele Male wiederholen.
4. Für eine neue Stichprobe gibt jeder Baum eine Stimme ab -> man wählt die **Mehrheitsentscheidung**.

**Warum es funktioniert:** Jeder Baum neigt auf eine leicht unterschiedliche Weise zum Overfitting, da jeder eine andere Bootstrap-Stichprobe betrachtet. Die Mehrheitsentscheidung mittelt die einzelnen Fehler aus, wodurch die **Varianz** reduziert wird, während der **Bias** annähernd gleich bleibt.

In [ ]:
def bagging_predictions(x_train, y_train, x_test, tree_count: int, seed: int = seed) -> np.ndarray:
    generator = np.random.default_rng(seed)
    n, n_test = x_train.shape[0], x_test.shape[0]
    result = np.zeros((tree_count, n_test), dtype=np.uint16)

    for index_tree in range(tree_count):
        # 1. Bootstrap-Stichprobe
        indices_samples = generator.choice(n, size=n, replace=True)
        x_bootstrap, y_bootstrap = x_train[indices_samples], y_train[indices_samples]

        # 2. Baum trainieren
        tree = DecisionTreeClassifier(random_state=generator.integers(low=0, high=10000))
        tree.fit(x_bootstrap, y_bootstrap)

        # 3. Vorhersage
        result[index_tree, :] = tree.predict(x_test)
    return result

# Genauigkeit eines einzelnen Entscheidungsbaums berechnen
single_tree = DecisionTreeClassifier(random_state=seed)
single_tree.fit(x_train, y_train)
y_hat = single_tree.predict(x_test)
accuracy_tree = accuracy_score(y_test, y_hat)

# Genauigkeit einen Baggings-Modells von 50 Bäumen berechnen
predictions = bagging_predictions(x_train, y_train, x_test, tree_count=100)
y_hat = np.empty(x_test.shape[0], dtype=np.uint16)
for i in range(x_test.shape[0]):
    # Mehrheitsentscheidung der ersten 50 Bäume
    y_hat[i] = np.argmax(np.bincount(predictions[0:50, i]))
accuracy_bagging = accuracy_score(y_test, y_hat)

print(f'Genauigkeit 1 Entscheidungsbaum: {accuracy_tree:.3f}')
print(f' Genauigkeit 50 Bäume (Bagging): {accuracy_bagging:.3f}')

## Wie viele Bäume brauchen wir

Sehen wir uns an, wie sich die Genauigkeit verändert, wenn wir dem Ensemble weitere Bäume hinzufügen.

In [ ]:
# Genauigkeit bei wachsenden Ensemblegrößen messen
accuracies = np.empty(predictions.shape[0], dtype=np.float64)
for k in range(1, predictions.shape[0] + 1):
    y_hat = np.empty(x_test.shape[0], dtype=np.uint16)
    for i in range(x_test.shape[0]):
        y_hat[i] = np.argmax(np.bincount(predictions[0:k, i]))
    accuracies[k - 1] = accuracy_score(y_test, y_hat)

figure, ax = plt.subplots(figsize=(7, 3), dpi=100)
ax.plot(np.arange(predictions.shape[0]) + 1, accuracies, color='steelblue', linewidth=1.5)
ax.axhline(accuracy_tree, color='indianred', linestyle='--', label='1 Baum')
ax.set(axisbelow=True, xlabel='Anzahl der Bäume im Ensemble', ylabel='Testgenauigkeit')
ax.set(xlim=[1, predictions.shape[0]])
ax.legend()
ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
figure.tight_layout()
plt.show(figure)

## Random Forest

Ein **Random Forest**-Modell besteht aus Bagging plus einem zusätzlichen Kniff:

> Bei jeder Aufspaltung berücksichtigt der Baum lediglich eine **zufällige Teilmenge der Merkmale** (typischerweise $\sqrt{p}$ Merkmale von insgesamt $p$).

Dies entkorreliert die einzelnen Bäume. Ohne diesen Schritt könnten alle Bäume an der Spitze dasselbe „dominante“ Merkmal auswählen und sich letztlich stark ähneln – was den Nutzen der Abstimmung verringern würde.

Die Modellklasse in sklearn ist [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html).

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,       # Anzahl der Bäume
    max_features='sqrt',    # Teilmenge von Merkmalen pro Split
    oob_score=True,         # Out-of-Bag-Score berechnen
    random_state=seed)
rf.fit(x_train, y_train)

accuracy_rf = rf.score(x_test, y_test)
print(f'Testgenauigkeit 1 Entscheidungsbaum: {accuracy_tree:.3f}')
print(f'            Testgenauigkeit CV-Baum: {accuracy_cv_tree:.3f}')
print(f' Testgenauigkeit 50 Bäume (Bagging): {accuracy_bagging:.3f}')
print(f'      Testgenauigkeit Random Forest: {accuracy_rf:.3f}')

## Out-of-Bag-Fehler (OOB-Fehler)

Jede Bootstrap-Stichprobe lässt ca. 36,8 % der Trainingsdaten aus. Bei jeder Trainingsstichprobe haben einige Bäume diese während des Trainings nicht berücksichtigt. Wir können diese Bäume verwenden, um die Stichprobe vorherzusagen – dies liefert uns eine Genauigkeitsschätzung, ohne dass ein separater Validierungsdatensatz oder eine Kreuzvalidierung erforderlich ist.

[sklearn](https://scikit-learn.org/stable/) berechnet dies automatisch, wenn wir `oob_score=True` setzen.

In [ ]:
print(f'Testsatz-Genauigkeit: {accuracy_rf:.3f}')
print(f'     OOB-Genauigkeit: {rf.oob_score_:.3f}')

## Merkmalswichtigkeit

Random Forests liefern ein natürliches Maß für die Merkmalswichtigkeit: Für jedes Merkmal wird die Gesamtreduktion der Unreinheit (Gini-Koeffizient) über alle Aufteilungen in allen Bäumen, in denen dieses Merkmal verwendet wurde, berechnet und auf $1$ normiert.

Dies zeigt uns: Auf welche klinischen Faktoren stützt sich der Random Forest am stärksten bei der Vorhersage von Herzerkrankungen?

In [ ]:
importances = rf.feature_importances_
sorted_idx = np.argsort(importances)

figure, ax = plt.subplots(figsize=(6, 6), dpi=100)
ax.barh(range(len(sorted_idx)), importances[sorted_idx])
ax.set(axisbelow=True, xlabel='Merkmalswichtigkeit (Gini)')
ax.set(ylim=[-0.5, sorted_idx.size - 0.5], yticks=range(len(sorted_idx)))
ax.set_yticklabels([feature_names_encoded[i] for i in sorted_idx])
ax.grid(axis='x', color='#A0A0A0', linestyle='--', linewidth=0.5)
figure.tight_layout()
plt.show(figure)